

# Exploración de datos con Pandas (_Python Data Analysis_)

Francisco Jurado

<francisco.jurado@uam.es>

---

## Importando datos desde fuentes externas

In [ ]:
import pandas as pd

Para poder importar archivos `xls` necesitaremos instalar una librería adicional para que pandas pueda leer su contenido

In [ ]:
%pip install xlrd

Podrá importar datos desde los formatos más habituales (csv, tsv, excel, html, json, etc.), que se encuentren accesibles en su disco o en la red.

En el siguiente ejemplo se usa el nombre del fichero local...

In [ ]:
dataset_location = "titanic3.xls"

... pero puede ser la ruta a una url remota

In [ ]:
dataset_location = "https://hbiostat.org/data/repo/titanic3.xls"

Eche un vistazo al contenido del archivo para entender su estructura y https://hbiostat.org/data/repo/ctitanic3 para más detalles.

Para cargar el dataset en un `dataframe`, basta con usar el método correspondiente de la familia `pd.read_*`, y si fuera necesario, adaptar mediante parámetros las opciones de lectura.

In [ ]:
df = pd.read_excel(dataset_location)

In [ ]:
help(pd.read_excel)

Echemos un vistazo a las primeras filas del dataframe.

In [ ]:
df.head(10)

Veamos cuáles son las dimensiones del dataframe observando su _forma_ (shape)

In [ ]:
df.shape

Si lo que se desea es acceder a información concreta, bastará usar a la propiedad o al método correspondiente.

Veamos cuáles son las columnas de las que dispone el dataframe.

In [ ]:
df.columns

## Analizando y recodificando las columnas

Para obtener más información acerca de las columnas, tipos de datos, etc. se dispone del método `info()`

In [ ]:
df.info()

Si solo deseamos conocer de qué tipo son los datos almacenados en las columnas, podemos acceder al atributo `dtypes`.

In [ ]:
df.dtypes

Como puede apreciarse, la columna `sex` está almacenada como un objeto (una cadena de caracteres), y la columna `survived` (si sobrevivió o no) está almacenada como entero. 

Procedamos a cambiarlas como datos categóricos. Para ello puede emplearse el método `astype`.

In [ ]:
df.sex = df.sex.astype("category") # empleando el operador '.' para acceder a la colmna
df["survived"] = df["survived"].astype("bool") # empleando el operador "[]" para acceder a la columna

Echemos un vistazo ahora a los tipos de las columnas.

In [ ]:
df.dtypes

In [ ]:
df.head()

__Ejercicio:__ Convierta la columna `pclass` que indica la clase en la que viajaba cada pasajero en una categoría en lugar de seguir empleandola como entero.

In [ ]:
# Escribe tu código aquí


Para analizar los diferentes valores que toma una variable categórica, por ejemplo `embarked`, se puede emplear el método `unique()` de la siguiente forma:

In [ ]:
df.embarked.unique()

Mirando la documentación del dataset del Titanic, se puede leer que la variable `embarked` toma los valores 'C', 'Q' y 'S', que corresponden a los puertos de embarque Cherbourg, Queenstown y Southampton, respectivamente.

Pasemos a recodificar estos valores por su nombre completo para que resulte más significativo.

In [ ]:
df.embarked = df.embarked.replace({'C': 'Cherbourg', 'Q': 'Queenstown', 'S': 'Southampton'})

¿De qué tipo tipo de dato de df.embarked?


In [ ]:
df.embarked.dtype.name

Una vez reemplazadas las cadenas de caracteres por otras más significativas, convirtamos la columna 'embarked' en una categoría.


In [ ]:
df.embarked = df.embarked.astype("category")

Fíjese cómo ha cambiado el resultado del método `unique()` tras la recodificación:

In [ ]:
df.embarked.unique()

Para no tener que ir una a una, puede comprobarse qué valores toman **todas las columnas que están codificadas como "Object"**.

Basta un bucle `for` que recorra las columnas del DataFrame que sean de tipo "Object".

Para ello, puede hacerse uso del método `select_dtypes`, que permite seleccionar columnas según su tipo de dato.

In [ ]:
for col in df.select_dtypes(include=['object','str']).columns: # str es diferente a object desde pandas 2.0
    print(f"Columna '{col}': {df[col].unique()}")

Atendiendo a lo que albergan las columnas:

- 'name' que tiene el nombre de la persona.
- 'ticket' con el número de billete. 
- 'cabin' con el número de camarote.
- 'boat' con el número de bote salvavidas.
- 'home.dest' con el destino final de la persona.
- 'body' con el número de identificación o del cuerpo en caso de fallecimiento.

Excepto 'body', el resto de las columnas contienen texto libre, por lo que no es aconsejable convertirlas en categorías.

Las demás columnas 'object' pueden convertirse en categorías. Hagámoslo con un bucle.

In [ ]:
except_cols = ['name', 'ticket', 'cabin', 'boat', 'home.dest']

for col in df.select_dtypes(include=['object', 'str']).columns.difference(except_cols):
    df[col] = df[col].astype('category')

In [ ]:
df.dtypes

Ya se ha visto que `body` contiene números enteros. Opciones:

- Podría dejarse como entera (¿tendría sentido?), 
- convertirse en categoría (¿tendría sentido?), o 
- convertirse en cadena de caracteres.

In [ ]:
df.body = df.body.astype('str') # o bien 'str', o 'category', o 'Int64' (entero que admite NaN)

In [ ]:
df.info()

¿Y cuánta memoria está usando cada columna en bytes?

Para eso se dispone del método `memory_usage()`, el cual devuelve una Serie con el uso de memoria de cada columna del DataFrame.

Este método recibir el parámetro `deep=True` para obtener una estimación más precisa del uso de memoria, especialmente para columnas de tipo objeto o categórico.

In [ ]:
df.memory_usage(deep=True)

Para hacernos una primera idea sobre los datos almacenados en el dataframe, el método `describe()` proporciona algo de *"información estadística descriptiva"* que puede resultar útil.

In [ ]:
df.describe()

Fíjese que no aparecen aquellos datos definidos como categoríass, cadenas de caracteres, etc., sino solo los datos almacenados como numéricos.

__Ejercicio:__ Vuelva a ejecutar las anteriores celdas de este `notebook` pero esta vez sin realizar la conversión a tipo categórico de las columnas.

__Pregunta:__ ¿Qué hubiera ocurrido al no convertir a tipo de dato categórigo la columna `survived`, `pclass`, `sex` o `body`?

In [ ]:
df.survived.unique(), df.sex.unique(), df.pclass.unique(), df.embarked.unique()

En lugar de haber recodificado `survived` como booleano, podríamos haberlo recodificado como 'No sobrevivió', 'Sobrevivió'.

In [ ]:
df.survived.map({False: 'No sobrevivió', True: 'Sobrevivió'}).astype('category')

¡Ojo!, esto no modifica el DataFrame original. Hay que asignarlo de nuevo si queremos guardar los cambios:

Aunque también podríamos haber convertido primero a categoría y luego renombrado las categorías:

In [ ]:
df.survived = df.survived.astype('category').cat.rename_categories({0: 'No sobrevivió', 1: 'Sobrevivió'})

In [ ]:
df.head()

## Identificación y tratamiento de valores nulos

Los valores nulos se refieren a datos que faltan o no están disponibles en un DataFrame o una Series.

Generalmente se representan mediante `NaN` (Not a Number).

Estos valores representan la ausencia de un valor y pueden causar problemas en operaciones (p.ej. estadísticas), por lo que es necesario identificarlos y manejarlos adecuadamente:

- eliminando las filas/columnas que los contengan
- reemplazándolos: 
    - media en datos numéricos
    - moda en datos categóricos
- asignándoles un valor concreto si la semántica lo permita

Para obtener los valores nulos, bastará con invocar:

In [ ]:
df.isna()

... o bien:

In [ ]:
df.isnull() # es un alias de df.isna(), es decir, que el método isnull() invoca internamente a isna()

In [ ]:
df.isna().sum()

Para identificar las variables, si las hay, que tienen valores nulos e indicar cuántos nulos tiene cada una de ellas, se puede hacer lo siguiente.

In [ ]:
df.isna().sum()[df.isna().sum() > 0]

Algunas alternativas:

In [ ]:
df.isna().sum()[df.isna().any()]

In [ ]:
df.isna().sum()[lambda x: x > 0]

In [ ]:
df.isna().sum().loc[lambda x: x > 0]

In [ ]:
pd.DataFrame([(col, df[col].isna().sum()) for col in df.columns if df[col].isna().sum() > 0], columns=['Column', 'Missing Values'])

Puede comprobarse cómo las variables `body` y `cabin` tiene 1188 y 1014 valores nulos respectivamente de 1309.

¿Cómo podría eliminarlas?

In [ ]:
df.drop(columns=["body", "cabin"], inplace=True)
# alternativamente: df = df.drop(columns=["body", "cabin"])

Por su parte, `fare` y `embarked` tienen muy pocos valores nulos, por lo que podríamos optar por eliminar las filas que los contengan:

In [ ]:
df.dropna(subset=['fare'], inplace=True)
# alternativamente: df = df.dropna(subset=['fare'])

In [ ]:
df.isna().sum()[df.isna().any()]

¿Por qué no he eliminado `embarked`?

Como se ha adelantado, en ocasiones podría ser interesante reemplazar los valores nulos de una variable categórica por el valor más frecuente (moda).
A modo de ejemplo, hagámoslo con la columna `embarked`.

In [ ]:
most_frequent = df.embarked.mode()[0]
df.embarked = df.embarked.fillna(most_frequent)

Y en el caso de una variable continua (p.ej. `age`) se podrían reemplazar los nulos por la media:

In [ ]:
mean_age = df.age.mean()

# no ejecute la siguiente línea, pues reemplaza todos los nulos por la media global
# df.age = df.age.fillna(mean_age) 

# en su lugar, use esta otra para ver el resultado sin modificar el DataFrame original.
df['age'].fillna(mean_age) 

__Pregunta:__ ¿Es una buena elección hacer estas atribuciones?

Otra opción para el tratamiento de los valores nulos, es asignándoles un valor concreto que encaje con la semántica de la variable.

A modo de ejemplo, para la columna `boat`, que indica el número de bote salvavidas asignado a cada pasajero, se podría considerar que la ausencia de valor indica que 
el pasajero no fue asignado a ningún bote salvavidas, por lo que podemos rellenar los valores nulos con una cadena de texto como 'Not_assigned'.

In [ ]:
df.boat = df.boat.fillna('Not_assigned').astype('category') # podemos convertir a categoría tras rellenar los nulos, o dejarla como objeto.

Algo parecido pasa con home.dest, que tiene muchos nulos y no parece práctico eliminarlos. 

Simplemente, se desconoce el destino del pasajero, por lo que podemos asignarles un valor `Unknown`.

In [ ]:
df['home.dest'] = df['home.dest'].fillna('Unknown').astype('category')

__Pregunta:__ ¿Por qué he usado `df['home.dest']` en lugar de `df.home.dest`? ¿Daría igual usar uno u otro?

Veamos cómo queda ahora el resumen de valores nulos:

In [ ]:
df.isna().sum()[df.isna().any()]

Para la edad, que es una variable numérica, tiene sentido reemplazar los valores nulos por la media de la columna. 

Sin embargo, como veremos más adelante, la edad puede ser diferente según el sexo y la clase en la que viajaba el pasajero, por lo que una opción más adecuada sería calcular la media de edad para cada combinación de sexo y clase, y usar esos valores para rellenar los nulos.

## Agrupando información (`groupby`)

Supongamos que se desea agrupar los pasajeros en grupos según el sexo.

Emplearemos el método `groupby` que crea atendiendo a los diferentes valores que se encuentren en la columna o columas seleccionadas.

In [ ]:
df.groupby("sex")

El argumento `observed=False` se usa para que se muestren todas las categorías, incluso aquellas que no aparecen en el DataFrame (en este caso, no hay ningún pasajero de sexo 'other', pero si hubieran existido, con `observed=False` se mostraría la categoría 'other' con un conteo de 0)

In [ ]:
df.groupby("sex", observed=False)

Este objeto `DataFrameGroupBy` consiste en una lista de tuplas conteniendo pares `(clave, dataframe)` donde:

- _clave_: es un valor único dentro de la columna seleccionada
- _dataframe_: es el dataframe resultante de cada agrupación
    
Veámoslo más claro con algo de código:

In [ ]:
grouped_df = df.groupby("sex")  # agrupamos por sexo

¿Cuántos grupos se han creado?

In [ ]:
grouped_df.ngroups

Ahora, para cada grupo, veamos la clave y el dataset con la agrupación.

In [ ]:
for key, gr in grouped_df:      # para cada par (clave, dataframe)
    print("Grupo: ", key)       # veamos la clave
    print(gr.info())            # veamos la info. del dataframe

Cualquier operación que se aplique sobre el objeto de tipo `DataFrameGroupBy`, se realizará **sobre cada uno de los grupos**. 

Así, podríamos obtener estadística descriptiva por grupo...

In [ ]:
grouped_df.describe()

Transpongamos el resultado:

In [ ]:
grouped_df.describe().T

... o acceder a una de las series (columna) específicas, y aplicar sobre ella la operación pertinente para cada grupo.

In [ ]:
grouped_df["age"].describe().T

Si lo que se desea es trabajar con un grupo concreto, puede extraerse el `DataFrame` específico para dicho grupo.

In [ ]:
grouped_df.get_group("female")

Esta aproximación permite aplicar, sobre cada grupo, el método de agregación que deseemos. 

Por ejemplo, supongamos que se desea conocer el número de mujeres y hombres que viajaban por clase.

In [ ]:
df.groupby("sex").size()

Y si se desea conocer el número de pasajeros por clase.

In [ ]:
df.groupby("pclass").size()

Y la media de edad y pasaje por clase.

In [ ]:
df.loc[:,["pclass", "age", "fare"]].groupby("pclass").mean()

Incluso, se pueden crear agrupaciones por valores de más de una columna.

Por ejemplo, calculemos la media de edad por sexo y clase.

In [ ]:
df.groupby(['sex', 'pclass'])['age'].mean()

Ahora se entenderá mejor, por qué no era buena idea asignar la media global de la edad para rellenar los valores nulos de la edad, ya que la media de edad varía significativamente entre los diferentes grupos definidos por el sexo y la clase del pasajero.

Con esto, se podrían aplicar transformaciones para rellenar los valores nulos de la edad con la media de la edad dentro de cada grupo (combinación de sexo y clase):

In [ ]:
df['age'] = df.groupby(['sex', 'pclass'])['age'].transform(lambda x: x.fillna(x.mean()))

El método `transform` permite aplicar una función y devolver un objeto que tiene el mismo índice que el DataFrame original (lo que es útil para asignar los resultados de vuelta a una columna del DataFrame original).

Para que se entienda mejor el uso de `transform`, veamos un ejemplo sencillo creando un DataFrame de ejemplo y sin agrupaciones complejas.


In [ ]:
df_example = pd.DataFrame({
    'A': ['foo', 'bar', 'foo', 'bar', 'foo', 'bar'],
    'B': [1, 2, 3, 4, 5, 6],
    'C': [10, 20, 30, 40, 50, 60]
})  
df_example.A.transform(lambda x: x + '_suffix')

In [ ]:
df_example.loc[:, ['B', 'C']].transform(lambda x: x - x.mean())

Aprovechando estos datos para introducir algunos conceptos de _plotting_.

In [ ]:
%pip install --quiet matplotlib

In [ ]:
%matplotlib inline

df.groupby("sex").size().plot.pie(legend=True, title="Distribución por sexo")

Fíjese sobre qué tipo de información se ejecuta `plot`.

In [ ]:
type(df.groupby("sex").size())

In [ ]:
help(pd.Series.plot)

In [ ]:
df.groupby('sex').size().plot(kind='pie', title='Mi primera tarta', grid=True)

In [ ]:
df.groupby('sex').size().plot.pie(title='Otra tarta', xlabel='Sexo', legend=True)

In [ ]:
help(pd.plotting)

In [ ]:
help(pd.DataFrame.plot)

In [ ]:
df.groupby("sex").size().plot(kind="bar")

In [ ]:
df.groupby("sex").size().plot(kind="pie")

Hecha esta pequeña introducción a cómo visualizar información de forma gráfica, volvamos a las agrupaciones.

Supongamos que ahora se quiere conocer el número de mujeres y hombres que viajaban por clase. 

Basta con indicar la lista de columnas en el parámetro pasado a `groupby`.

In [ ]:
df.groupby(["pclass", "sex"]).size()

In [ ]:
df.groupby(["pclass", "sex"]).size().plot(kind="bar")

In [ ]:
df.groupby(["pclass", "sex"]).size().unstack(1)

In [ ]:
df.groupby(["pclass", "sex"]).size().unstack(1).plot(kind="bar")

In [ ]:
help(pd.DataFrame.unstack)

In [ ]:
df.groupby(["pclass", "sex"]).size().unstack(1).plot(kind="bar", stacked=True)

## Agregación de datos (`aggregate`)

Además de los métodos como `describe`, `size`, etc. que realizan algunas operaciones de conteo y estadística descriptiva, se pueden aplicar agregaciones de datos más complejas mediante los métodos `agg` (alias de `aggregate`), el cual recibe como parámetro la función (o funciones) a aplicar sobre el conjunto de datos.

In [ ]:
help(df.agg)

In [ ]:
df.loc[:,["fare", "age"]].mean()

In [ ]:
df.loc[:,["fare", "age"]].max()

In [ ]:
df.loc[:,["fare", "age"]].count()

In [ ]:
df.loc[:,["fare", "age"]].agg(['mean', 'max'])

In [ ]:
df.loc[:,["fare", "age"]].agg(['mean', 'max']).T

In [ ]:
df.loc[:,["fare", "age"]].agg(["mean", "min", "max"])

También pueden aplicarse sobre agrupaciones.

In [ ]:
df.loc[:,["pclass", "sex", "age", "fare"]].groupby(["pclass", "sex"]).agg(["mean", "min", "max"])

La lista de funciones que recibe por parámetro puede ser cualquier función, como por ejemplo, las implementadas en NumPy
o funciones lambda que nosotros definamos.

In [ ]:
import numpy as np 

df.loc[:,["pclass", "sex", "age"]].groupby(["pclass", "sex"]).agg([np.mean, np.std, np.min, np.max])

Este mensaje de `FutureWarning` significa que estamos utilizando .mean(), .std(), .min(), o .max() en un objeto SeriesGroupBy de Pandas de una manera que se modificará
en futuras versiones de la biblioteca.

La advertencia informa que en el futuro deberá pasar una función directamente a la lógica/código.

In [ ]:
df.loc[:,["pclass", "sex", "age"]].groupby(["pclass", "sex"]).agg([lambda x: x.mean(), lambda x: x.std(), lambda x: x.min(), lambda x: x.max()])

¿Cuándo usar una agregación o un método predefinido?
Depende de la complejidad de la función a aplicar y de la disponibilidad de método o función predefinidos.

Por ejemplo, ¿cómo contaríamos el número de supervivientes por clase?

In [ ]:
df.loc[:,["pclass", "survived"]].groupby(["pclass", "survived"]).size()

... o empleando un agregado empleando

In [ ]:
df.loc[:,["pclass", "survived"]].groupby(["pclass", "survived"]).agg("size")

In [ ]:
type(df.loc[:,["pclass", "survived"]].groupby(["pclass", "survived"]).agg("size"))

__Ejercicio__ Por cada clase, calcule la media de edad de los pasajeros que sobrevivieron y los que no.

In [ ]:
df.loc[:,['pclass', 'age', 'survived']].groupby(['pclass','survived']).agg('mean')

## Más sobre gráficas y estadística descriptiva

Recordemos cómo obtener la media de la edad por clase y sexo:

In [ ]:
df.loc[:,["pclass", "sex", "age"]].groupby(["pclass", "sex"]).mean()

Podría obtenerse toda la estadística descriptiva.

In [ ]:
df.loc[:,["pclass", "sex", "age"]].groupby(["pclass", "sex"]).describe()

Aprovechemos aquí para introducir otro tipo de gráfico: un boxplot, que representa la distribución de una variable numérica mostrando sus cuartiles y posibles valores atípicos.

Usémoslo para representar la media de edad por clase.

In [ ]:
df.loc[:,["pclass", "age"]].boxplot("age", "pclass")

In [ ]:
help(pd.DataFrame.boxplot)

Otro tipo de gráfico que puede ser útil es el histograma. Este representa la distribución de una variable numérica dividiendo su rango en intervalos (o "bins") y 
mostrando la frecuencia de los valores que caen dentro de cada intervalo.

Veamos un ejemplo utilizando la variable "age" del conjunto de datos Titanic:

In [ ]:
df.loc[:,["pclass", "age"]].groupby("pclass")["age"].plot(kind="hist", alpha=0.5)

A pesar del alpha, sigue parece algo incómodo ver la información. Veámoslo mejor para cada grupo de datos concreto.

In [ ]:
df.loc[:,"age"].hist(by=df["pclass"])

In [ ]:
help(pd.Series.hist)